In [19]:
import os

from dotenv import load_dotenv
load_dotenv()

VAST_API_KEY = os.getenv("VAST_AI_API_KEY")
GITHUB_PAT = os.getenv("GITHUB_PAT")

In [23]:
import requests
import json
import subprocess

exclude_machines = {22768}

def search():
    url = "https://console.vast.ai/api/v0/search/asks/"

    payload = json.dumps({
        "q": {
            "order": [[
                "dph_total", "asc"
            ]],
            "verified": {"eq": True},
            "rentable": {"eq": True},
            "type": "on-demand",
            "allocated_storage": "16",
            "reliability2": {"gt": 0.995},
            "inet_down": {"gt": 1000},
            "inet_up": {"gt": 1000},
            "duration": {"gte": 1},
            "inet_up_cost": {"lte": 0.001},
            "inet_down_cost": {"lte": 0.001},
            "geolocation": {"in": ["US", "CA"]},
            "dph_total": {"lte": 0.2},
        }
    })
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }

    response = requests.request("PUT", url, headers=headers, data=payload)
    response.raise_for_status()
    offers = response.json()["offers"]

    return [o for o in offers if o["machine_id"] not in exclude_machines]

def create_template(offer_id):
    url = f"https://console.vast.ai/api/v0/asks/{offer_id}/"
    payload = json.dumps({
        "template_id": "251090"
    })
    headers = {
        'Authorization': f'Bearer {VAST_API_KEY}',
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }
    response = requests.request("PUT", url, headers=headers, data=payload)
    response.raise_for_status()
    return response.json()

def get_instance(instance_id):
    url = f"https://console.vast.ai/api/v0/instances/{instance_id}/"

    payload = {}
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {VAST_API_KEY}'
    }

    response = requests.request("GET", url, headers=headers, data=payload)
    response.raise_for_status()
    return response.json()["instances"]

def get_ssh_command(instance_id):
    result = subprocess.run(["/Users/shivamsarodia/.pyenv/shims/vastai", "ssh-url", str(instance_id)], 
                            capture_output=True, text=True, check=True)
    ssh_url = result.stdout.strip()
    return f"ssh -i ~/.ssh/id_ed25519_personal {ssh_url}"

In [24]:
# Do a search and get a server

results = search()
top = results[0]
print("Cost:\t", top["dph_total"])
print("ID:\t", results[0]["id"])

Cost:	 0.09096296296296297
ID:	 19639098


In [25]:
# Create an instance using this template

result = create_template(top["id"])
instance_id = result["new_contract"]

In [28]:
import time

while True:
    time.sleep(10)
    print(get_instance(instance_id)["actual_status"])

KeyboardInterrupt: 

In [32]:
get_ssh_command(instance_id)

'ssh -i ~/.ssh/id_ed25519_personal ssh://root@136.60.217.200:9188'

In [ ]:
# Set up git
git config --global user.email "ssarodia@gmail.com"
git config --global user.name "Shivam Sarodia"
git clone https://github.com/ShivamSarodia/AlphaBlokus.git

cd AlphaBlokus

# So we can push to the repo
git remote set-url origin https://$GITHUB_PAT@github.com/ShivamSarodia/AlphaBlokus.git

# Install poetry if not already installed
if ! command -v poetry &> /dev/null; then
    curl -sSL https://install.python-poetry.org | python3 -
    export PATH="$HOME/.local/bin:$PATH"
fi

poetry install

In [ ]:
# Add whatever is needed to be able to Github pull/push
# ^ https://chatgpt.com/share/686621bd-bd4c-800a-9669-f08c6c723b14
# Clone the repo
# Cd into the repo
# Install poetry
# Run poetry install
# Copy the moves_20.npz over (or maybe just check it into the repo? idk)
# Print SSH command